In [ ]:
# | default_exp gsc_client


# GSC Client
> Google Search Console authentication, date range helpers, and data fetching utilities.

In [ ]:
# | export
import pickle
from datetime import datetime, timedelta
from pathlib import Path
from fastcore.basics import store_attr

from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request as GoogleRequest

In [ ]:
# | export
class GSCAuth:
    "Google Search Console authentication handler."
    SCOPES = [
        "https://www.googleapis.com/auth/webmasters.readonly",
        "https://www.googleapis.com/auth/webmasters",
    ]

    def __init__(self,
                 secrets_file: str = "./client_secrets.json",  # Path to OAuth client secrets
                 token_file: str = "./token.pickle"  # Path to cached token
                 ):
        store_attr()
        self.token_file = Path(token_file)
        self.credentials = self._load_credentials()

    def _load_credentials(self) -> Credentials | None:
        "Load credentials from token file if it exists."
        if not self.token_file.exists(): return None
        with open(self.token_file, "rb") as f: return pickle.load(f)

    def _save_credentials(self, credentials: Credentials):
        "Persist credentials to token file."
        with open(self.token_file, "wb") as f: pickle.dump(credentials, f)

    def get_credentials(self) -> Credentials | None:
        "Return valid credentials, refreshing if expired."
        if not self.credentials:
            raise ValueError("No credentials found. Run `authenticate()` first.")

        if self.credentials.expired:
            self.credentials.refresh(GoogleRequest())
            self._save_credentials(self.credentials)
        return self.credentials

    def authenticate(self):
        "Run OAuth flow to obtain and save new credentials."
        flow = InstalledAppFlow.from_client_secrets_file(self.secrets_file, scopes=self.SCOPES)
        self.credentials = flow.run_local_server(port=0)
        self._save_credentials(self.credentials)

In [ ]:
# | hide
#| eval: false
auth = GSCAuth(secrets_file="client_secrets.json")
auth.authenticate()

In [ ]:
# | export
def get_date_range(range_type: str = "today",
                   # One of: today, this_week, last_week, this_month, last_month, last_7_days, last_days, last_months, entire_history, custom
                   days: int | None = None,  # Number of days for `last_days`
                   months: int | None = None,  # Number of months for `last_months`
                   start_date: str | None = None,  # Start date for `custom` (YYYY-MM-DD)
                   end_date: str | None = None,  # End date for `custom` (YYYY-MM-DD)
                   ) -> tuple[str, str]:
    "Generate a date range for GSC queries, accounting for the 3-day data delay."
    fmt = lambda d: d.strftime("%Y-%m-%d")
    today = datetime.now()
    latest = today - timedelta(days=3)

    match range_type:
        case "today":
            return fmt(latest), fmt(latest)
        case "last_7_days":
            return fmt(latest - timedelta(days=7)), fmt(latest)
        case "this_week":
            return fmt(today - timedelta(days=today.weekday())), fmt(latest)
        case "this_month":
            return fmt(today.replace(day=1)), fmt(latest)
        case "last_week":
            start = today - timedelta(days=today.weekday() + 7)
            return fmt(start), fmt(start + timedelta(days=6))
        case "last_month":
            end = today.replace(day=1) - timedelta(days=1)
            return fmt(end.replace(day=1)), fmt(end)
        case "entire_history":
            return "2020-01-01", fmt(latest)
        case "last_days" if days:
            return fmt(latest - timedelta(days=days)), fmt(latest)
        case "last_months" if months:
            return fmt(latest - timedelta(days=30 * months)), fmt(latest)
        case "custom" if start_date and end_date:
            s = datetime.strptime(start_date, "%Y-%m-%d")
            e = min(datetime.strptime(end_date, "%Y-%m-%d"), latest)
            return fmt(s), fmt(e)
        case _:
            raise ValueError(f"Invalid range_type or missing parameters: {range_type!r}")



In [ ]:
# | export
def fetch_gsc_data(auth: GSCAuth,  # Authenticated GSCAuth instance
                   site_url: str,  # GSC property URL
                   start_date: str,  # Start date (YYYY-MM-DD)
                   end_date: str,  # End date (YYYY-MM-DD)
                   dimensions: list[str] | None = None,  # GSC dimensions to group by
                   row_limit: int = 25000  # Max rows to fetch
                   ) -> list[dict]:
    "Fetch analytics rows from Google Search queryConsole."
    if dimensions is None: dimensions = ["query", "page", "country", "device"]
    service = build("searchconsole", "v1", credentials=auth.get_credentials())
    body = {"startDate": start_date, "endDate": end_date, "dimensions": dimensions, "rowLimit": row_limit}
    return service.searchanalytics().query(siteUrl=site_url, body=body).execute().get("rows", [])


In [ ]:
# | export
def get_verified_sites(auth: GSCAuth  # Authenticated GSCAuth instance
                       ) -> list[dict]:
    "Get all verified GSC properties for the authenticated account."
    service = build("searchconsole", "v1", credentials=auth.get_credentials())
    sites = service.sites().list().execute().get("siteEntry", [])
    return [{"site_url": s["siteUrl"], "permission_level": s["permissionLevel"]}
            for s in sites if s["permissionLevel"] != "siteUnverifiedUser"]


In [ ]:
# | hide
#| eval: false

sites = get_verified_sites(auth)
sites


In [ ]:
# | hide
#| eval: false
from pprint import pprint

start, end = get_date_range("last_days", 1)
data = fetch_gsc_data(auth, "sc-domain:kareemai.com", start, end)
pprint(data)
          'https://kareemai.com/blog/posts/products_reviews/Huawei%20freebuds%207i.html',
           'hun',
           'MOBILE'],